# foundation

In [1]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOOD DATABASE BUILDER
# ============================================================
#
# Purpose:
#   Build a small, auditable nutrition database from USDA
#   FoodData Central Foundation Foods CSV files.
#
# Features:
#   - Validates USDA CSV files and required columns
#   - Detects Foundation Foods robustly
#   - Normalizes USDA data_type values
#   - Detects duplicate FDC ID + nutrient ID rows
#   - Safely resolves duplicate nutrient rows
#   - Uses unique-FDC-ID nutrient coverage
#   - Explicitly selects the energy nutrient
#   - Performs exact normalized food-description matching
#   - Handles duplicate food descriptions deterministically
#   - Validates selected descriptions
#   - Preserves missing nutrients as NaN
#   - Builds a 100-g nutrition database
#   - Performs macro-energy comparison
#   - Saves complete audit files
#   - Reloads the saved database and validates it
#   - Runs a sample nutrition calculation
#
# IMPORTANT:
#   Missing nutrient != zero.
#
#   NaN means USDA did not provide a usable value for the
#   selected nutrient in that food record.
#
# ============================================================

import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

USDA_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30"
    r"\FoodData_Central_foundation_food_csv_2026-04-30"
)

FOOD_CSV = USDA_DIR / "food.csv"
FOOD_NUTRIENT_CSV = USDA_DIR / "food_nutrient.csv"
NUTRIENT_CSV = USDA_DIR / "nutrient.csv"


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

DATABASE_CSV = USDA_DIR / "food_database.csv"
SOURCE_MAPPING_CSV = USDA_DIR / "food_source_mapping.csv"
FOOD_SELECTION_REVIEW_CSV = USDA_DIR / "food_selection_review.csv"
NUTRIENT_MAPPING_CSV = USDA_DIR / "nutrient_mapping.csv"
DUPLICATE_NUTRIENT_REVIEW_CSV = (
    USDA_DIR / "duplicate_food_nutrient_review.csv"
)


# ============================================================
# TARGET FOODS
# ============================================================
#
# The exact USDA description is deliberately used.
#
# DO NOT replace these with fuzzy matching.
#
# If USDA changes the description in a future data release,
# the pipeline should fail rather than silently select another
# food.
# ============================================================

TARGET_FOODS = {

    "chicken_breast": {
        "exact_description":
            "Chicken, breast, boneless, skinless, raw"
    },

    "beef_ground": {
        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw"
    },

    "oats_rolled": {
        "exact_description":
            "Oats, whole grain, rolled, old fashioned"
    },

    "carrot": {
        "exact_description":
            "Carrots, mature, raw"
    },

    "rye_flour": {
        "exact_description":
            "Flour, rye"
    },

    "apple_fuji": {
        "exact_description":
            "Apples, fuji, with skin, raw"
    },

    "banana": {
        "exact_description":
            "Bananas, ripe and slightly ripe, raw"
    },

    "milk_whole": {
        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D"
    },

    "mushroom_white_button": {
        "exact_description":
            "Mushrooms, white button"
    },
}


# ============================================================
# APPLICATION NUTRIENT POLICY
# ============================================================
#
# These are USDA nutrient IDs, but the application names are
# our stable internal names.
#
# Energy is selected separately because multiple KCAL
# nutrients exist in Foundation Foods.
# ============================================================

CRITICAL_NUTRIENTS = {
    "protein_g": {
        "nutrient_id": 1003,
        "expected_unit": "G",
        "expected_name": "Protein",
    },

    "fat_g": {
        "nutrient_id": 1004,
        "expected_unit": "G",
        "expected_name": "Total lipid (fat)",
    },

    "carbohydrate_g": {
        "nutrient_id": 1005,
        "expected_unit": "G",
        "expected_name": "Carbohydrate, by difference",
    },

    "fiber_g": {
        "nutrient_id": 1079,
        "expected_unit": "G",
        "expected_name": "Fiber, total dietary",
    },

    "sugar_g": {
        "nutrient_id": 1063,
        "expected_unit": "G",
        "expected_name": "Sugars, Total",
    },

    "saturated_fat_g": {
        "nutrient_id": 1258,
        "expected_unit": "G",
        "expected_name": "Fatty acids, total saturated",
    },
}


# ------------------------------------------------------------
# Energy policy
# ------------------------------------------------------------
#
# Select the KCAL nutrient with the highest number of UNIQUE
# Foundation Food records containing a valid value.
#
# If coverage ties:
#
#   2047 > 2048 > 1008
#
# This is an explicit application policy.
# ============================================================

ENERGY_PRIORITY = {
    2047: 3,   # Energy (Atwater General Factors)
    2048: 2,   # Energy (Atwater Specific Factors)
    1008: 1,   # Energy
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def print_section(title):
    """Print a consistent section heading."""

    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def print_subsection(title):
    """Print a consistent subsection heading."""

    print()
    print("-" * 70)
    print(title)
    print("-" * 70)


def normalize_text(value):
    """
    Normalize text for exact comparison.

    This is NOT fuzzy matching.

    It:
      - converts NaN to empty string
      - Unicode normalizes
      - lowercases
      - converts punctuation/separators to spaces
      - collapses whitespace
    """

    if pd.isna(value):
        return ""

    value = str(value)

    value = unicodedata.normalize(
        "NFKC",
        value
    )

    value = value.lower().strip()

    # Convert punctuation/separators to spaces.
    value = re.sub(
        r"[^a-z0-9%]+",
        " ",
        value
    )

    # Collapse whitespace.
    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def require_columns(df, required_columns, filename):
    """Fail if required columns are missing."""

    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{filename} is missing required columns: "
            f"{missing}"
        )


def to_numeric_series(series):
    """Safely convert a pandas Series to numeric."""

    return pd.to_numeric(
        series,
        errors="coerce"
    )


# ============================================================
# START
# ============================================================

print_section(
    "LOADING USDA FOODDATA CENTRAL FILES"
)

print("Search directory:")
print(USDA_DIR)


# ============================================================
# FILE EXISTENCE CHECK
# ============================================================

print_subsection(
    "CHECKING USDA FILES"
)

required_files = [
    FOOD_CSV,
    FOOD_NUTRIENT_CSV,
    NUTRIENT_CSV,
]

for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required USDA file not found:\n{path}"
        )

    print(
        f"PASS: {path.name}"
    )


# ============================================================
# SELECTED FILES
# ============================================================

print_subsection(
    "SELECTED USDA FILES"
)

print("food.csv:")
print(FOOD_CSV)

print("food_nutrient.csv:")
print(FOOD_NUTRIENT_CSV)

print("nutrient.csv:")
print(NUTRIENT_CSV)


# ============================================================
# READ CSV FILES
# ============================================================

print_section(
    "READING USDA CSV FILES"
)

food = pd.read_csv(
    FOOD_CSV,
    low_memory=False
)

food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_CSV,
    low_memory=False
)

nutrient = pd.read_csv(
    NUTRIENT_CSV,
    low_memory=False
)

print(
    f"food.csv rows:          {len(food):,}"
)

print(
    f"food_nutrient.csv rows: {len(food_nutrient):,}"
)

print(
    f"nutrient.csv rows:      {len(nutrient):,}"
)


# ============================================================
# USDA COLUMN VALIDATION
# ============================================================

print_section(
    "USDA COLUMN VALIDATION"
)

require_columns(
    food,
    [
        "fdc_id",
        "description",
        "data_type",
    ],
    "food.csv"
)

require_columns(
    food_nutrient,
    [
        "fdc_id",
        "nutrient_id",
        "amount",
    ],
    "food_nutrient.csv"
)

require_columns(
    nutrient,
    [
        "id",
        "name",
        "unit_name",
    ],
    "nutrient.csv"
)

print(
    "food.csv columns: PASS"
)

print(
    "food_nutrient.csv columns: PASS"
)

print(
    "nutrient.csv columns: PASS"
)


# ============================================================
# NORMALIZE DATA TYPES
# ============================================================

print_section(
    "USDA DATA TYPES"
)

print(
    food["data_type"]
    .value_counts(dropna=False)
    .to_string()
)


food["normalized_data_type"] = (
    food["data_type"]
    .astype("string")
    .str.lower()
    .str.replace(
        r"[^a-z0-9]+",
        "",
        regex=True
    )
)

print_subsection(
    "NORMALIZED DATA TYPES"
)

print(
    food["normalized_data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# FOUNDATION FOOD DETECTION
# ============================================================

foundation_food = food[
    food["normalized_data_type"]
    == "foundationfood"
].copy()


if foundation_food.empty:

    print()
    print(
        "FOUNDATION FOOD DETECTION FAILED"
    )

    print(
        food["normalized_data_type"]
        .value_counts(dropna=False)
        .to_string()
    )

    raise RuntimeError(
        "No Foundation Foods were found."
    )


# ============================================================
# FOUNDATION FOOD SUMMARY
# ============================================================

print_section(
    "FOUNDATION FOOD SUMMARY"
)

print(
    f"Foundation Foods found: "
    f"{len(foundation_food):,}"
)

foundation_food_ids = set(
    foundation_food["fdc_id"]
    .astype(int)
)

print(
    f"Unique Foundation FDC IDs: "
    f"{len(foundation_food_ids):,}"
)

if len(foundation_food_ids) != len(foundation_food):

    duplicate_food_ids = (
        foundation_food[
            foundation_food["fdc_id"]
            .duplicated(keep=False)
        ]
        .sort_values("fdc_id")
    )

    print(
        duplicate_food_ids[
            [
                "fdc_id",
                "description",
                "data_type",
            ]
        ]
        .to_string(index=False)
    )

    raise RuntimeError(
        "Duplicate Foundation Food FDC IDs detected."
    )


# ============================================================
# PREPARE FOUNDATION NUTRIENTS
# ============================================================

foundation_food_nutrients = (
    food_nutrient[
        food_nutrient["fdc_id"]
        .isin(foundation_food_ids)
    ]
    .copy()
)


print_section(
    "FOUNDATION NUTRIENT COVERAGE"
)

print(
    f"Foundation nutrient rows: "
    f"{len(foundation_food_nutrients):,}"
)

print(
    f"Nutrient IDs available in Foundation Foods: "
    f"{foundation_food_nutrients['nutrient_id'].nunique():,}"
)


# ============================================================
# DUPLICATE FOOD/NUTRIENT CHECK
# ============================================================

print_section(
    "FOOD/NUTRIENT DUPLICATE CHECK"
)

duplicate_nutrient_rows = (
    foundation_food_nutrients
    .groupby(
        [
            "fdc_id",
            "nutrient_id",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

duplicate_nutrient_rows = (
    duplicate_nutrient_rows[
        duplicate_nutrient_rows["count"] > 1
    ]
    .copy()
)

if duplicate_nutrient_rows.empty:

    print(
        "PASS: No duplicate FDC ID + nutrient ID combinations."
    )

else:

    print(
        "WARNING:",
        len(duplicate_nutrient_rows),
        "duplicate food/nutrient combinations found."
    )

    print(
        duplicate_nutrient_rows
        .to_string(index=False)
    )


# ============================================================
# RESOLVE DUPLICATE FOOD/NUTRIENT RECORDS
# ============================================================
#
# Safe policy:
#
#   1. No duplicate:
#        keep row.
#
#   2. Duplicate rows with identical numeric amount:
#        collapse to one row.
#
#   3. Duplicate rows with conflicting amounts:
#        FAIL.
#
# We do NOT blindly use values.iloc[0].
# ============================================================

print_section(
    "RESOLVING FOOD/NUTRIENT RECORDS"
)


def resolve_duplicate_food_nutrients(df):
    """
    Canonicalize duplicate fdc_id + nutrient_id rows.

    Identical nutrient amounts are considered safe duplicates.

    Conflicting nutrient amounts cause the pipeline to fail,
    because silently choosing one would be unsafe.
    """

    key_columns = [
        "fdc_id",
        "nutrient_id",
    ]

    amount_numeric = (
        pd.to_numeric(
            df["amount"],
            errors="coerce"
        )
    )

    working = df.copy()

    working["_amount_numeric"] = amount_numeric

    conflicting_records = []

    canonical_rows = []

    for (
        (fdc_id, nutrient_id),
        group
    ) in working.groupby(
        key_columns,
        sort=False,
        dropna=False
    ):

        numeric_values = (
            group["_amount_numeric"]
            .dropna()
            .unique()
        )

        # ----------------------------------------------------
        # No duplicate
        # ----------------------------------------------------

        if len(group) == 1:

            row = group.iloc[0].copy()

            canonical_rows.append(
                row
            )

            continue

        # ----------------------------------------------------
        # All amounts missing
        # ----------------------------------------------------

        if len(numeric_values) == 0:

            row = group.iloc[0].copy()

            canonical_rows.append(
                row
            )

            continue

        # ----------------------------------------------------
        # Multiple different numeric values
        # ----------------------------------------------------

        if len(numeric_values) > 1:

            conflict = group.copy()

            conflict[
                "conflict_fdc_id"
            ] = fdc_id

            conflict[
                "conflict_nutrient_id"
            ] = nutrient_id

            conflicting_records.append(
                conflict
            )

            continue

        # ----------------------------------------------------
        # Duplicate rows but same numeric amount
        # ----------------------------------------------------

        row = group.iloc[0].copy()

        canonical_rows.append(
            row
        )

    # --------------------------------------------------------
    # Fail on conflicts
    # --------------------------------------------------------

    if conflicting_records:

        conflicts = pd.concat(
            conflicting_records,
            ignore_index=True
        )

        review_columns = [
            col
            for col in [
                "fdc_id",
                "nutrient_id",
                "amount",
                "data_points",
                "derivation_id",
                "min",
                "max",
                "median",
                "footnote",
            ]
            if col in conflicts.columns
        ]

        conflicts[
            review_columns
        ].to_csv(
            DUPLICATE_NUTRIENT_REVIEW_CSV,
            index=False
        )

        print()
        print(
            "ERROR: Conflicting duplicate nutrient values detected."
        )

        print(
            conflicts[
                review_columns
            ]
            .to_string(index=False)
        )

        print()
        print(
            "Conflict review saved to:"
        )

        print(
            DUPLICATE_NUTRIENT_REVIEW_CSV
        )

        raise RuntimeError(
            "Conflicting duplicate FDC ID + nutrient ID "
            "records detected. Pipeline stopped."
        )

    # --------------------------------------------------------
    # Build canonical table
    # --------------------------------------------------------

    canonical = pd.DataFrame(
        canonical_rows
    )

    canonical = (
        canonical
        .drop(
            columns=[
                "_amount_numeric"
            ],
            errors="ignore"
        )
        .reset_index(drop=True)
    )

    return canonical


foundation_food_nutrients_canonical = (
    resolve_duplicate_food_nutrients(
        foundation_food_nutrients
    )
)


# ============================================================
# CANONICAL TABLE VALIDATION
# ============================================================

duplicate_after_resolution = (
    foundation_food_nutrients_canonical
    .duplicated(
        subset=[
            "fdc_id",
            "nutrient_id",
        ]
    )
    .any()
)


if duplicate_after_resolution:

    raise RuntimeError(
        "Canonical nutrient table still contains duplicate "
        "FDC ID + nutrient ID combinations."
    )


print(
    "Canonical nutrient table created:"
)

print(
    f"Rows: "
    f"{len(foundation_food_nutrients_canonical):,}"
)

print(
    "Unique FDC ID + nutrient ID combinations: PASS"
)


# ============================================================
# NUTRIENT LOOKUP
# ============================================================

nutrient_lookup = (
    nutrient[
        [
            "id",
            "name",
            "unit_name",
        ]
    ]
    .copy()
)

nutrient_lookup["id"] = (
    pd.to_numeric(
        nutrient_lookup["id"],
        errors="coerce"
    )
)

nutrient_lookup = (
    nutrient_lookup
    .dropna(subset=["id"])
    .copy()
)

nutrient_lookup["id"] = (
    nutrient_lookup["id"]
    .astype(int)
)


# ============================================================
# MERGE NUTRIENT METADATA
# ============================================================

foundation_food_nutrients_canonical = (
    foundation_food_nutrients_canonical
    .merge(
        nutrient_lookup,
        left_on="nutrient_id",
        right_on="id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_nutrient")
    )
)


# ============================================================
# UNIQUE-FDC-ID NUTRIENT COVERAGE
# ============================================================

def nutrient_coverage(nutrient_id):
    """
    Return the number of UNIQUE Foundation Foods with a
    usable numeric value for the specified nutrient.
    """

    rows = (
        foundation_food_nutrients_canonical[
            foundation_food_nutrients_canonical[
                "nutrient_id"
            ]
            == int(nutrient_id)
        ]
    )

    if rows.empty:
        return 0

    valid_foods = rows[
        pd.to_numeric(
            rows["amount"],
            errors="coerce"
        ).notna()
    ]

    return int(
        valid_foods["fdc_id"].nunique()
    )


# ============================================================
# ENERGY NUTRIENTS
# ============================================================

print_section(
    "ENERGY NUTRIENTS ACTUALLY AVAILABLE"
)

energy_candidates = []

for nutrient_id, priority in ENERGY_PRIORITY.items():

    matching = nutrient_lookup[
        nutrient_lookup["id"]
        == nutrient_id
    ]

    if matching.empty:
        continue

    nutrient_row = (
        matching.iloc[0]
    )

    coverage = nutrient_coverage(
        nutrient_id
    )

    energy_candidates.append(
        {
            "id": nutrient_id,
            "name": nutrient_row["name"],
            "unit_name": nutrient_row["unit_name"],
            "coverage": coverage,
            "priority": priority,
        }
    )


energy_candidates_df = pd.DataFrame(
    energy_candidates
)


if energy_candidates_df.empty:

    raise RuntimeError(
        "No supported KCAL energy nutrients were found."
    )


energy_candidates_df = (
    energy_candidates_df
    .sort_values(
        [
            "coverage",
            "priority",
        ],
        ascending=[
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)


print(
    energy_candidates_df
    .to_string(index=False)
)


# ============================================================
# SELECT ENERGY NUTRIENT
# ============================================================

selected_energy = (
    energy_candidates_df.iloc[0]
)

ENERGY_NUTRIENT_ID = int(
    selected_energy["id"]
)

ENERGY_NUTRIENT_NAME = (
    selected_energy["name"]
)

ENERGY_NUTRIENT_UNIT = (
    selected_energy["unit_name"]
)

ENERGY_COVERAGE = int(
    selected_energy["coverage"]
)


if ENERGY_NUTRIENT_UNIT.upper() != "KCAL":

    raise RuntimeError(
        "Selected energy nutrient is not KCAL."
    )


# ============================================================
# BUILD FINAL NUTRIENT MAP
# ============================================================

NUTRIENT_MAP = {

    "calories_kcal": {
        "nutrient_id": ENERGY_NUTRIENT_ID,
        "expected_unit": "KCAL",
        "expected_name": ENERGY_NUTRIENT_NAME,
    },
}


for application_name, config in CRITICAL_NUTRIENTS.items():

    NUTRIENT_MAP[
        application_name
    ] = config


# ============================================================
# FINAL NUTRIENT MAP DISPLAY
# ============================================================

print_section(
    "FINAL FOUNDATION NUTRIENT MAP"
)

nutrient_mapping_rows = []

for application_name, config in NUTRIENT_MAP.items():

    nutrient_id = int(
        config["nutrient_id"]
    )

    matching = nutrient_lookup[
        nutrient_lookup["id"]
        == nutrient_id
    ]

    if matching.empty:

        raise RuntimeError(
            f"Required nutrient ID not found: "
            f"{nutrient_id}"
        )

    nutrient_row = (
        matching.iloc[0]
    )

    actual_name = str(
        nutrient_row["name"]
    )

    actual_unit = str(
        nutrient_row["unit_name"]
    )

    coverage = nutrient_coverage(
        nutrient_id
    )

    print(
        f"{application_name:<25} "
        f"ID={nutrient_id:>5} "
        f"UNIT={actual_unit:<5} "
        f"COVERAGE={coverage:>4} "
        f"NAME={actual_name}"
    )

    nutrient_mapping_rows.append(
        {
            "application_name":
                application_name,

            "USDA_nutrient_id":
                nutrient_id,

            "USDA_name":
                actual_name,

            "unit":
                actual_unit,

            "coverage":
                coverage,
        }
    )


nutrient_mapping_df = pd.DataFrame(
    nutrient_mapping_rows
)


# ============================================================
# NUTRIENT MAPPING VALIDATION
# ============================================================

print_section(
    "NUTRIENT MAPPING VALIDATION"
)

for application_name, config in NUTRIENT_MAP.items():

    nutrient_id = int(
        config["nutrient_id"]
    )

    matching = nutrient_lookup[
        nutrient_lookup["id"]
        == nutrient_id
    ]

    if matching.empty:

        raise RuntimeError(
            f"Missing nutrient ID: "
            f"{nutrient_id}"
        )

    nutrient_row = (
        matching.iloc[0]
    )

    actual_name = str(
        nutrient_row["name"]
    )

    actual_unit = str(
        nutrient_row["unit_name"]
    )

    expected_name = str(
        config["expected_name"]
    )

    expected_unit = str(
        config["expected_unit"]
    )

    if actual_name != expected_name:

        raise RuntimeError(
            f"NUTRIENT NAME VALIDATION FAILED: "
            f"{application_name}: "
            f"expected '{expected_name}', "
            f"got '{actual_name}'"
        )

    if actual_unit.upper() != expected_unit.upper():

        raise RuntimeError(
            f"NUTRIENT UNIT VALIDATION FAILED: "
            f"{application_name}: "
            f"expected '{expected_unit}', "
            f"got '{actual_unit}'"
        )


# Ensure no two application fields accidentally use
# the same nutrient ID.

used_nutrient_ids = [
    int(config["nutrient_id"])
    for config in NUTRIENT_MAP.values()
]

if len(used_nutrient_ids) != len(
    set(used_nutrient_ids)
):

    raise RuntimeError(
        "Two or more application nutrient fields "
        "use the same USDA nutrient ID."
    )


print(
    "NUTRIENT MAPPING VALIDATION: PASS"
)


# ============================================================
# FOUNDATION FOOD TARGET SEARCH
# ============================================================

print_section(
    "FOUNDATION FOOD TARGET SEARCH"
)

foundation_food = foundation_food.copy()

foundation_food[
    "normalized_description"
] = (
    foundation_food["description"]
    .apply(normalize_text)
)


target_matches = {}

for food_name, config in TARGET_FOODS.items():

    expected_description = (
        config["exact_description"]
    )

    normalized_expected = (
        normalize_text(
            expected_description
        )
    )

    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == normalized_expected
    ].copy()

    target_matches[
        food_name
    ] = matches

    print_subsection(
        food_name.upper()
    )

    print(
        "Requested:"
    )

    print(
        expected_description
    )

    print(
        f"Matches: {len(matches)}"
    )

    if not matches.empty:

        print(
            matches[
                [
                    "fdc_id",
                    "description",
                ]
            ]
            .sort_values("fdc_id")
            .to_string(index=False)
        )

    if matches.empty:

        raise RuntimeError(
            f"Food not found: "
            f"{food_name}: "
            f"{expected_description}"
        )


# ============================================================
# SELECT TARGET FOUNDATION FOODS
# ============================================================

print_section(
    "SELECTING TARGET FOUNDATION FOODS"
)


def requested_nutrient_completeness(
    fdc_id
):
    """
    Count how many requested application nutrients have
    non-null values for this food.

    This includes:
      calories_kcal
      protein_g
      fat_g
      carbohydrate_g
      fiber_g
      sugar_g
      saturated_fat_g
    """

    selected_nutrient_ids = [
        int(config["nutrient_id"])
        for config in NUTRIENT_MAP.values()
    ]

    rows = (
        foundation_food_nutrients_canonical[
            foundation_food_nutrients_canonical[
                "fdc_id"
            ]
            == int(fdc_id)
        ]
    )

    if rows.empty:
        return 0

    rows = rows[
        rows["nutrient_id"]
        .isin(selected_nutrient_ids)
    ]

    valid_count = (
        pd.to_numeric(
            rows["amount"],
            errors="coerce"
        )
        .notna()
        .sum()
    )

    return int(
        valid_count
    )


selected_foods = {}
food_selection_review_rows = []


for food_name, config in TARGET_FOODS.items():

    matches = (
        target_matches[
            food_name
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Single exact match
    # --------------------------------------------------------

    if len(matches) == 1:

        row = matches.iloc[0]

        fdc_id = int(
            row["fdc_id"]
        )

        selected_foods[
            food_name
        ] = row

        print_subsection(
            food_name.upper()
        )

        print(
            "STATUS: SELECTED"
        )

        print(
            f"FDC ID: {fdc_id}"
        )

        print(
            f"Description: "
            f"{row['description']}"
        )

        food_selection_review_rows.append(
            {
                "food_name":
                    food_name,

                "fdc_id":
                    fdc_id,

                "description":
                    row["description"],

                "match_count":
                    len(matches),

                "requested_nutrient_completeness":
                    requested_nutrient_completeness(
                        fdc_id
                    ),

                "selection_reason":
                    "Unique exact normalized description match",
            }
        )

        continue

    # --------------------------------------------------------
    # Duplicate exact description
    # --------------------------------------------------------

    matches[
        "requested_nutrient_completeness"
    ] = (
        matches["fdc_id"]
        .apply(
            requested_nutrient_completeness
        )
    )

    # Highest completeness first.
    #
    # FDC ID ascending is ONLY a deterministic tie-breaker.
    matches = (
        matches
        .sort_values(
            [
                "requested_nutrient_completeness",
                "fdc_id",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )

    selected_row = (
        matches.iloc[0]
    )

    selected_fdc_id = int(
        selected_row["fdc_id"]
    )

    selected_foods[
        food_name
    ] = selected_row

    print_subsection(
        food_name.upper()
    )

    print(
        "STATUS: DUPLICATE_DESCRIPTION"
    )

    print(
        matches[
            [
                "fdc_id",
                "description",
                "requested_nutrient_completeness",
            ]
        ]
        .to_string(index=False)
    )

    print()
    print(
        f"Selected preferred FDC ID: "
        f"{selected_fdc_id}"
    )

    print(
        "Reason: highest requested nutrient "
        "completeness; FDC ID used as "
        "deterministic tie-breaker."
    )

    for _, review_row in matches.iterrows():

        food_selection_review_rows.append(
            {
                "food_name":
                    food_name,

                "fdc_id":
                    int(review_row["fdc_id"]),

                "description":
                    review_row["description"],

                "match_count":
                    len(matches),

                "requested_nutrient_completeness":
                    int(
                        review_row[
                            "requested_nutrient_completeness"
                        ]
                    ),

                "selection_reason":
                    (
                        "Highest requested nutrient "
                        "completeness; FDC ID used "
                        "as deterministic tie-breaker"
                    ),
            }
        )


food_selection_review_df = pd.DataFrame(
    food_selection_review_rows
)


# ============================================================
# SELECTED FOOD DESCRIPTION VALIDATION
# ============================================================

print_section(
    "SELECTED FOOD DESCRIPTION VALIDATION"
)

for food_name, config in TARGET_FOODS.items():

    if food_name not in selected_foods:

        raise RuntimeError(
            f"Selected food missing: "
            f"{food_name}"
        )

    selected = (
        selected_foods[
            food_name
        ]
    )

    actual = normalize_text(
        selected["description"]
    )

    expected = normalize_text(
        config["exact_description"]
    )

    if actual != expected:

        raise RuntimeError(
            f"DESCRIPTION VALIDATION FAILED: "
            f"{food_name}: "
            f"expected '{expected}', "
            f"got '{actual}'"
        )

    print(
        f"PASS: {food_name:<30} "
        f"-> {selected['description']}"
    )


# ============================================================
# BUILD FINAL FOOD DATABASE
# ============================================================

print_section(
    "BUILDING FINAL FOOD DATABASE"
)


def get_food_nutrients(
    fdc_id
):
    """
    Return application nutrient values for one FDC ID.

    Missing nutrients remain NaN.

    This function operates only on the canonical nutrient table,
    so duplicate nutrient rows cannot silently overwrite values.
    """

    rows = (
        foundation_food_nutrients_canonical[
            foundation_food_nutrients_canonical[
                "fdc_id"
            ]
            == int(fdc_id)
        ]
    )

    result = {}

    for application_name, config in NUTRIENT_MAP.items():

        nutrient_id = int(
            config["nutrient_id"]
        )

        matching = rows[
            rows["nutrient_id"]
            == nutrient_id
        ]

        if matching.empty:

            result[
                application_name
            ] = np.nan

            continue

        if len(matching) != 1:

            raise RuntimeError(
                f"Canonical nutrient table has "
                f"unexpected duplicate rows: "
                f"FDC {fdc_id}, "
                f"nutrient {nutrient_id}"
            )

        value = pd.to_numeric(
            matching.iloc[0]["amount"],
            errors="coerce"
        )

        if pd.isna(value):

            result[
                application_name
            ] = np.nan

        else:

            result[
                application_name
            ] = float(value)

    return result


database_rows = []

source_mapping_rows = []


for food_name, config in TARGET_FOODS.items():

    selected = (
        selected_foods[
            food_name
        ]
    )

    fdc_id = int(
        selected["fdc_id"]
    )

    description = str(
        selected["description"]
    )

    nutrient_values = (
        get_food_nutrients(
            fdc_id
        )
    )

    row = {

        "food_name":
            food_name,

        "fdc_id":
            fdc_id,

        "description":
            description,

        "data_type":
            "Foundation Food",

        "basis_g":
            100,
    }

    row.update(
        nutrient_values
    )

    database_rows.append(
        row
    )

    source_mapping_rows.append(
        {
            "food_name":
                food_name,

            "fdc_id":
                fdc_id,

            "description":
                description,

            "data_type":
                "Foundation Food",

            "basis_g":
                100,

            "source_file":
                str(FOOD_CSV),

            "food_nutrient_source_file":
                str(FOOD_NUTRIENT_CSV),

            "nutrient_source_file":
                str(NUTRIENT_CSV),

            "energy_nutrient_id":
                ENERGY_NUTRIENT_ID,

            "energy_nutrient_name":
                ENERGY_NUTRIENT_NAME,
        }
    )


food_database = pd.DataFrame(
    database_rows
)


source_mapping_df = pd.DataFrame(
    source_mapping_rows
)


# ============================================================
# COLUMN ORDER
# ============================================================

database_columns = [

    "food_name",
    "fdc_id",
    "description",
    "data_type",
    "basis_g",

    "calories_kcal",

    "protein_g",
    "fat_g",
    "carbohydrate_g",
    "fiber_g",
    "sugar_g",
    "saturated_fat_g",
]


food_database = (
    food_database[
        database_columns
    ]
)


# ============================================================
# FINAL FOOD DATABASE DISPLAY
# ============================================================

print_section(
    "FINAL FOOD DATABASE"
)

print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# NUTRIENT DATABASE SANITY CHECK
# ============================================================

print_section(
    "NUTRIENT DATABASE SANITY CHECK"
)

for application_name, config in NUTRIENT_MAP.items():

    nutrient_id = int(
        config["nutrient_id"]
    )

    matching = nutrient_lookup[
        nutrient_lookup["id"]
        == nutrient_id
    ]

    if matching.empty:

        raise RuntimeError(
            f"Missing nutrient ID: "
            f"{nutrient_id}"
        )

    nutrient_row = (
        matching.iloc[0]
    )

    print(
        f"PASS: {application_name:<25} "
        f"-> {nutrient_row['name']}"
    )


print()
print(
    "No nutrient IDs are incorrectly reused."
)


# ============================================================
# NUTRIENT COVERAGE COUNTS
# ============================================================

nutrient_columns = list(
    NUTRIENT_MAP.keys()
)

for column in nutrient_columns:

    coverage_count = (
        food_database[column]
        .notna()
        .sum()
    )

    print(
        f"Foods with {column}: "
        f"{coverage_count}/"
        f"{len(food_database)}"
    )


# ============================================================
# NEGATIVE NUTRIENT CHECK
# ============================================================

negative_records = []

for column in nutrient_columns:

    numeric_values = pd.to_numeric(
        food_database[column],
        errors="coerce"
    )

    mask = (
        numeric_values < 0
    )

    if mask.any():

        negative_rows = (
            food_database.loc[
                mask,
                [
                    "food_name",
                    "fdc_id",
                    column,
                ]
            ]
            .copy()
        )

        negative_records.append(
            negative_rows
        )


if negative_records:

    negative_df = pd.concat(
        negative_records,
        ignore_index=True
    )

    print()
    print(
        negative_df
        .to_string(index=False)
    )

    raise RuntimeError(
        "Negative nutrient values detected."
    )

else:

    print()
    print(
        "No negative nutrient values detected."
    )


# ============================================================
# 100-G BASIS VALIDATION
# ============================================================

if not (
    food_database["basis_g"]
    .eq(100)
    .all()
):

    raise RuntimeError(
        "100 g basis validation failed."
    )

print(
    "100 g basis validation: PASS"
)


# ============================================================
# FOUNDATION FOOD DATA TYPE VALIDATION
# ============================================================

if not (
    food_database["data_type"]
    .eq("Foundation Food")
    .all()
):

    raise RuntimeError(
        "Foundation Food data type validation failed."
    )

print(
    "Foundation Food data type validation: PASS"
)


# ============================================================
# CRITICAL NUTRIENT COVERAGE
# ============================================================

required_for_database = [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrate_g",
]

critical_missing = []

for column in required_for_database:

    if (
        food_database[column]
        .isna()
        .any()
    ):

        missing_foods = (
            food_database.loc[
                food_database[column].isna(),
                "food_name"
            ]
            .tolist()
        )

        critical_missing.append(
            (
                column,
                missing_foods
            )
        )


if critical_missing:

    for column, foods in critical_missing:

        print(
            f"WARNING: {column} missing for "
            f"{foods}"
        )

    raise RuntimeError(
        "Critical nutrient coverage failed."
    )

else:

    print(
        "Critical nutrient coverage: PASS"
    )


# ============================================================
# MACRO ENERGY COMPARISON
# ============================================================
#
# This is diagnostic only.
#
# It does NOT replace USDA calorie values.
#
# Simple calculation:
#
#   protein * 4
# + carbohydrate * 4
# + fat * 9
#
# USDA energy can differ because the actual USDA energy
# methodology is more specific than this simple calculation.
# ============================================================

print_section(
    "MACRO ENERGY COMPARISON"
)

macro_rows = []


for _, row in food_database.iterrows():

    protein = row["protein_g"]
    carbs = row["carbohydrate_g"]
    fat = row["fat_g"]
    calories = row["calories_kcal"]

    if any(
        pd.isna(value)
        for value in [
            protein,
            carbs,
            fat,
            calories,
        ]
    ):

        macro_calories = np.nan
        difference = np.nan

    else:

        macro_calories = (
            protein * 4
            + carbs * 4
            + fat * 9
        )

        difference = (
            calories
            - macro_calories
        )

    macro_rows.append(
        {
            "food_name":
                row["food_name"],

            "calories_kcal":
                calories,

            "macro_calorie_check":
                macro_calories,

            "macro_calorie_difference":
                difference,
        }
    )


macro_comparison_df = pd.DataFrame(
    macro_rows
)


print(
    macro_comparison_df
    .to_string(index=False)
)


# ============================================================
# DATABASE VALIDATION
# ============================================================

print_section(
    "DATABASE VALIDATION"
)

expected_food_names = set(
    TARGET_FOODS.keys()
)

actual_food_names = set(
    food_database["food_name"]
)

missing_food_names = (
    expected_food_names
    -
    actual_food_names
)

unexpected_food_names = (
    actual_food_names
    -
    expected_food_names
)


if missing_food_names:

    raise RuntimeError(
        f"Missing expected foods: "
        f"{sorted(missing_food_names)}"
    )


if unexpected_food_names:

    raise RuntimeError(
        f"Unexpected foods in database: "
        f"{sorted(unexpected_food_names)}"
    )


if (
    len(food_database)
    != len(TARGET_FOODS)
):

    raise RuntimeError(
        "Final food count does not match TARGET_FOODS."
    )


if (
    food_database["fdc_id"]
    .nunique()
    != len(food_database)
):

    raise RuntimeError(
        "Duplicate FDC IDs exist in final database."
    )


if (
    food_database["food_name"]
    .nunique()
    != len(food_database)
):

    raise RuntimeError(
        "Duplicate food names exist in final database."
    )


if not (
    food_database["basis_g"]
    .eq(100)
    .all()
):

    raise RuntimeError(
        "Not all foods use a 100 g basis."
    )


if not (
    food_database["data_type"]
    .eq("Foundation Food")
    .all()
):

    raise RuntimeError(
        "Not all records are Foundation Foods."
    )


# Validate every final description again.

for _, row in food_database.iterrows():

    food_name = row["food_name"]

    expected = normalize_text(
        TARGET_FOODS[
            food_name
        ]["exact_description"]
    )

    actual = normalize_text(
        row["description"]
    )

    if expected != actual:

        raise RuntimeError(
            f"Final description mismatch: "
            f"{food_name}"
        )


print(
    f"Foods: {len(food_database)}"
)

print(
    f"Unique FDC IDs: "
    f"{food_database['fdc_id'].nunique()}"
)

print(
    "Basis values:",
    food_database["basis_g"]
    .unique()
    .tolist()
)

print(
    "Data types:",
    food_database["data_type"]
    .unique()
    .tolist()
)

print(
    "Critical nutrient coverage: PASS"
)

print()
print(
    "STATUS: PASS"
)


# ============================================================
# SAVE DATABASE FILES
# ============================================================

print_section(
    "SAVING DATABASE FILES"
)


# ------------------------------------------------------------
# Database
# ------------------------------------------------------------

food_database.to_csv(
    DATABASE_CSV,
    index=False
)

print(
    "Database:"
)

print(
    DATABASE_CSV
)


# ------------------------------------------------------------
# Source mapping
# ------------------------------------------------------------

source_mapping_df.to_csv(
    SOURCE_MAPPING_CSV,
    index=False
)

print()
print(
    "Source mapping:"
)

print(
    SOURCE_MAPPING_CSV
)


# ------------------------------------------------------------
# Food selection review
# ------------------------------------------------------------

food_selection_review_df.to_csv(
    FOOD_SELECTION_REVIEW_CSV,
    index=False
)

print()
print(
    "Duplicate food review:"
)

print(
    FOOD_SELECTION_REVIEW_CSV
)


# ------------------------------------------------------------
# Nutrient mapping
# ------------------------------------------------------------

nutrient_mapping_df.to_csv(
    NUTRIENT_MAPPING_CSV,
    index=False
)

print()
print(
    "Nutrient mapping:"
)

print(
    NUTRIENT_MAPPING_CSV
)


# ------------------------------------------------------------
# Duplicate nutrient review
# ------------------------------------------------------------
#
# Even when duplicates are safely resolved, save the original
# duplicate records for audit purposes.
# ------------------------------------------------------------

if duplicate_nutrient_rows.empty:

    duplicate_review_df = pd.DataFrame(
        columns=[
            "fdc_id",
            "nutrient_id",
            "count",
        ]
    )

else:

    duplicate_review_df = (
        foundation_food_nutrients[
            foundation_food_nutrients[
                [
                    "fdc_id",
                    "nutrient_id",
                ]
            ]
            .apply(
                tuple,
                axis=1
            )
            .isin(
                duplicate_nutrient_rows[
                    [
                        "fdc_id",
                        "nutrient_id",
                    ]
                ]
                .apply(
                    tuple,
                    axis=1
                )
            )
        ]
        .copy()
    )


duplicate_review_df.to_csv(
    DUPLICATE_NUTRIENT_REVIEW_CSV,
    index=False
)

print()
print(
    "Duplicate nutrient review:"
)

print(
    DUPLICATE_NUTRIENT_REVIEW_CSV
)


# ============================================================
# RELOAD SAVED DATABASE
# ============================================================

print_section(
    "RELOADING SAVED DATABASE"
)

reloaded_database = pd.read_csv(
    DATABASE_CSV
)


# ------------------------------------------------------------
# Reload structure
# ------------------------------------------------------------

if list(
    reloaded_database.columns
) != database_columns:

    raise RuntimeError(
        "Saved database columns do not match "
        "expected schema."
    )


# ------------------------------------------------------------
# Reload count
# ------------------------------------------------------------

if len(
    reloaded_database
) != len(
    food_database
):

    raise RuntimeError(
        "Saved database row count changed after reload."
    )


# ------------------------------------------------------------
# Reload FDC IDs
# ------------------------------------------------------------

if set(
    reloaded_database["fdc_id"]
    .astype(int)
) != set(
    food_database["fdc_id"]
    .astype(int)
):

    raise RuntimeError(
        "Saved database FDC IDs do not match original."
    )


# ------------------------------------------------------------
# Reload food names
# ------------------------------------------------------------

if set(
    reloaded_database["food_name"]
) != set(
    food_database["food_name"]
):

    raise RuntimeError(
        "Saved database food names do not match original."
    )


print(
    "Saved database reload validation: PASS"
)


# ============================================================
# NUTRITION CALCULATION FUNCTIONS
# ============================================================

print_section(
    "NUTRITION CALCULATOR"
)


def nutrition_for_grams(
    food_name,
    grams,
    database=None
):
    """
    Calculate nutrition for a specified food amount.

    Database values are stored per 100 g.

    Missing nutrients remain NaN.

    Example:

        nutrition_for_grams(
            "chicken_breast",
            150
        )
    """

    if database is None:
        database = reloaded_database

    if food_name not in set(
        database["food_name"]
    ):

        raise KeyError(
            f"Unknown food: {food_name}"
        )

    grams = float(
        grams
    )

    if grams < 0:

        raise ValueError(
            "Food weight cannot be negative."
        )

    row = (
        database[
            database["food_name"]
            == food_name
        ]
        .iloc[0]
    )

    multiplier = (
        grams / 100.0
    )

    result = {}

    for nutrient_column in nutrient_columns:

        value = pd.to_numeric(
            row[nutrient_column],
            errors="coerce"
        )

        if pd.isna(value):

            result[
                nutrient_column
            ] = np.nan

        else:

            result[
                nutrient_column
            ] = float(
                value * multiplier
            )

    return result


def print_nutrition(
    nutrition
):
    """
    Print a nutrition result while preserving
    missing-vs-zero distinction.
    """

    display_names = {

        "calories_kcal":
            "calories_kcal",

        "protein_g":
            "protein_g",

        "fat_g":
            "fat_g",

        "carbohydrate_g":
            "carbohydrate_g",

        "fiber_g":
            "fiber_g",

        "sugar_g":
            "sugar_g",

        "saturated_fat_g":
            "saturated_fat_g",
    }

    for key in nutrient_columns:

        value = nutrition.get(
            key,
            np.nan
        )

        if pd.isna(value):

            display_value = "-"

        else:

            display_value = (
                f"{value:.2f}"
            )

        print(
            f"{display_names[key]:<25}: "
            f"{display_value}"
        )


# ============================================================
# EXAMPLE NUTRITION TEST
# ============================================================

print_subsection(
    "EXAMPLE NUTRITION TEST"
)

example_food = (
    "chicken_breast"
)

example_grams = 150.0

print()
print(
    f"{example_food} — "
    f"{example_grams:g} g"
)

example_nutrition = (
    nutrition_for_grams(
        example_food,
        example_grams
    )
)

print_nutrition(
    example_nutrition
)


# ============================================================
# MEAL CALCULATOR
# ============================================================

print_section(
    "MEAL"
)


meal = [

    (
        "chicken_breast",
        150.0
    ),

    (
        "oats_rolled",
        50.0
    ),

    (
        "apple_fuji",
        150.0
    ),
]


for food_name, grams in meal:

    print(
        f"{food_name:<25} "
        f"{grams:>8.1f} g"
    )


def calculate_meal(
    meal_items,
    database=None
):
    """
    Calculate total nutrition for a meal.

    meal_items format:

        [
            ("chicken_breast", 150),
            ("oats_rolled", 50),
            ("apple_fuji", 150),
        ]

    Missing nutrients are preserved.

    Important:
        If a nutrient is missing from every food,
        result is NaN.

        If a nutrient is present for at least one food,
        available values are summed while missing values
        contribute no numerical amount.

    This means:
        NaN != 0
    """

    if database is None:
        database = reloaded_database

    totals = {
        column: 0.0
        for column in nutrient_columns
    }

    present_any = {
        column: False
        for column in nutrient_columns
    }

    for food_name, grams in meal_items:

        nutrition = (
            nutrition_for_grams(
                food_name,
                grams,
                database
            )
        )

        for column in nutrient_columns:

            value = nutrition[
                column
            ]

            if pd.isna(value):

                continue

            present_any[
                column
            ] = True

            totals[
                column
            ] += float(value)

    # Convert nutrients that were unavailable for the
    # entire meal back to NaN.
    for column in nutrient_columns:

        if not present_any[column]:

            totals[column] = np.nan

    return totals


meal_totals = calculate_meal(
    meal
)


print()
print("-" * 70)
print("TOTAL NUTRITION")
print("-" * 70)

print_nutrition(
    meal_totals
)


# ============================================================
# FINAL PIPELINE SUMMARY
# ============================================================

print_section(
    "PIPELINE COMPLETE"
)

duplicate_food_count = (
    sum(
        len(
            matches
        ) > 1
        for matches in target_matches.values()
    )
)


print(
    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)

print(
    f"Foods selected: "
    f"{len(food_database):,}"
)

print(
    f"Foods requiring duplicate review: "
    f"{duplicate_food_count}"
)

print(
    "Foods not found: 0"
)

print()
print(
    "Database validation: PASS"
)

print()
print(
    "Database:"
)

print(
    DATABASE_CSV
)

print()
print(
    "Source mapping:"
)

print(
    SOURCE_MAPPING_CSV
)

print()
print(
    "Review:"
)

print(
    FOOD_SELECTION_REVIEW_CSV
)

print()
print(
    "Nutrient mapping:"
)

print(
    NUTRIENT_MAPPING_CSV
)

print()
print(
    "Duplicate nutrient review:"
)

print(
    DUPLICATE_NUTRIENT_REVIEW_CSV
)

print()
print(
    "Done."
)



LOADING USDA FOODDATA CENTRAL FILES
Search directory:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30

----------------------------------------------------------------------
CHECKING USDA FILES
----------------------------------------------------------------------
PASS: food.csv
PASS: food_nutrient.csv
PASS: nutrient.csv

----------------------------------------------------------------------
SELECTED USDA FILES
----------------------------------------------------------------------
food.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv
food_nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv
nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nut

# SR LEGACY

In [ ]:
# =============================================================================
# SR LEGACY STEP 11 — RELEASE CONSUMER / APPLICATION QUERY LAYER
# =============================================================================

from pathlib import Path
from datetime import datetime
import json
import hashlib
import re

import pandas as pd


# =============================================================================
# CONFIGURATION
# =============================================================================

STEP10_DIR = Path(
    r"C:\Users\AK\Downloads\zip\sr_legacy_step10"
)

STEP11_DIR = Path(
    r"C:\Users\AK\Downloads\zip\sr_legacy_step11"
)

RELEASE_DIR = STEP10_DIR / "release"

STEP11_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RELEASE_FILE = (
    RELEASE_DIR
    / "sr_legacy_ingredient_release.csv"
)

LOOKUP_FILE = (
    RELEASE_DIR
    / "sr_legacy_ingredient_release_lookup.csv"
)

SEARCH_FILE = (
    RELEASE_DIR
    / "sr_legacy_ingredient_release_search.csv"
)

MANIFEST_FILE = (
    STEP10_DIR
    / "sr_legacy_step10_release_manifest.json"
)


# =============================================================================
# HELPERS
# =============================================================================

def header(title):
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def normalize_text(value):
    """
    Conservative normalization for application search.

    Examples:

        "Spices, cinnamon, ground"
        ->
        "spices cinnamon ground"

        "Ground Cinnamon"
        ->
        "ground cinnamon"
    """

    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


def tokenize_text(value):
    """
    Convert text into searchable tokens.

    Example:

        "Spices, cinnamon, ground"
        ->
        ["spices", "cinnamon", "ground"]
    """

    normalized = normalize_text(value)

    if not normalized:
        return []

    return normalized.split()


def token_set(value):
    return set(
        tokenize_text(value)
    )


def bool_value(value):
    if pd.isna(value):
        return False

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
    }


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# =============================================================================
# SEARCH SCORING
# =============================================================================

def calculate_query_score(
    query,
    description,
    resolution_status=None,
):
    """
    Calculate consumer-search relevance.

    Ranking philosophy:

        Exact normalized description
            >
        Exact phrase
            >
        All query tokens in any order
            >
        Partial token coverage
            >
        Resolved records
            >
        Ingredient records

    This function does NOT modify the authoritative description.
    It only calculates search relevance.
    """

    q_normalized = normalize_text(
        query
    )

    d_normalized = normalize_text(
        description
    )

    if not q_normalized or not d_normalized:
        return 0.0

    q_tokens = token_set(
        q_normalized
    )

    d_tokens = token_set(
        d_normalized
    )

    if not q_tokens:
        return 0.0

    score = 0.0

    # -------------------------------------------------------------------------
    # EXACT DESCRIPTION
    # -------------------------------------------------------------------------

    if d_normalized == q_normalized:
        score += 1000

    # -------------------------------------------------------------------------
    # EXACT PHRASE
    # -------------------------------------------------------------------------

    if q_normalized in d_normalized:
        score += 800

    # -------------------------------------------------------------------------
    # TOKEN MATCHING
    # -------------------------------------------------------------------------

    matched_tokens = (
        q_tokens.intersection(
            d_tokens
        )
    )

    matched_count = len(
        matched_tokens
    )

    query_count = len(
        q_tokens
    )

    if matched_count == query_count:

        # Every query token exists.
        score += 700

    elif matched_count > 0:

        # Partial match.
        score += 500

    # -------------------------------------------------------------------------
    # TOKEN COVERAGE
    # -------------------------------------------------------------------------

    coverage = (
        matched_count / query_count
    )

    score += (
        coverage * 100
    )

    # -------------------------------------------------------------------------
    # MEANINGFUL TOKEN BONUS
    # -------------------------------------------------------------------------

    for token in matched_tokens:

        if len(token) >= 5:

            score += 10

        elif len(token) >= 3:

            score += 5

    # -------------------------------------------------------------------------
    # RESOLUTION STATUS
    # -------------------------------------------------------------------------

    if resolution_status is not None:

        status = str(
            resolution_status
        ).upper()

        if status == "RESOLVED":

            score += 20

        elif status == "AMBIGUOUS":

            score += 5

    return score


# =============================================================================
# START
# =============================================================================

build_started = datetime.now().isoformat(
    timespec="microseconds"
)

header(
    "SR LEGACY STEP 11 — RELEASE CONSUMER / APPLICATION QUERY LAYER"
)

print(
    f"Build started: {build_started}"
)

print()
print("STEP 10 DIRECTORY:")
print(STEP10_DIR)

print()
print("STEP 11 OUTPUT DIRECTORY:")
print(STEP11_DIR)


# =============================================================================
# CHECK STEP 10 RELEASE
# =============================================================================

header(
    "CHECKING STEP 10 RELEASE"
)

required = [
    RELEASE_FILE,
    LOOKUP_FILE,
    SEARCH_FILE,
]

for path in required:

    if not path.exists():

        print(
            f"FAIL: {path.name}"
        )

        raise SystemExit(
            f"Missing Step 10 release file: {path}"
        )

    print(
        f"PASS: {path.name}"
    )


# =============================================================================
# LOAD DATA
# =============================================================================

header(
    "LOADING RELEASE DATA"
)

df = pd.read_csv(
    RELEASE_FILE,
    low_memory=False
)

lookup_df = pd.read_csv(
    LOOKUP_FILE,
    low_memory=False
)

search_df = pd.read_csv(
    SEARCH_FILE,
    low_memory=False
)

print(
    f"Release rows: {len(df):,}"
)

print(
    f"Lookup rows:  {len(lookup_df):,}"
)

print(
    f"Search rows:  {len(search_df):,}"
)


# =============================================================================
# REQUIRED COLUMN VALIDATION
# =============================================================================

header(
    "VALIDATING REQUIRED COLUMNS"
)

required_release_columns = [
    "fdc_id",
    "description",
    "description_normalized",
    "food_class",
    "resolved_fdc_id",
    "resolved_description",
    "resolution_status",
    "resolution_score",
    "resolution_confidence",
    "resolution_reason",
    "application_ready",
    "is_application_ingredient",
    "self_resolution_pass",
]

missing_columns = [
    column
    for column in required_release_columns
    if column not in df.columns
]

if missing_columns:

    raise SystemExit(
        "Missing required Step 10 release columns: "
        + ", ".join(missing_columns)
    )

print(
    f"Required columns: {len(required_release_columns)}"
)

print(
    "Required columns: PASS"
)


# =============================================================================
# NORMALIZE IDENTIFIERS
# =============================================================================

header(
    "NORMALIZING IDENTIFIERS"
)

for frame in [
    df,
    lookup_df,
    search_df,
]:

    if "fdc_id" in frame.columns:

        frame["fdc_id"] = pd.to_numeric(
            frame["fdc_id"],
            errors="coerce"
        ).astype("Int64")

    if "resolved_fdc_id" in frame.columns:

        frame["resolved_fdc_id"] = pd.to_numeric(
            frame["resolved_fdc_id"],
            errors="coerce"
        ).astype("Int64")

    if "query" in frame.columns:

        frame["query_normalized"] = (
            frame["query"]
            .map(normalize_text)
        )

    if "description" in frame.columns:

        frame["description_normalized_consumer"] = (
            frame["description"]
            .map(normalize_text)
        )

    elif "description_normalized" in frame.columns:

        frame["description_normalized_consumer"] = (
            frame["description_normalized"]
            .fillna("")
            .astype(str)
            .map(normalize_text)
        )


print(
    "FDC ID normalization: PASS"
)

print(
    "Text normalization: PASS"
)


# =============================================================================
# BUILD CONSUMER FIELDS
# =============================================================================

header(
    "BUILDING CONSUMER APPLICATION FIELDS"
)

df["consumer_fdc_id"] = (
    df["fdc_id"]
)

df["consumer_description"] = (
    df["description"]
)

df["consumer_description_normalized"] = (
    df["description_normalized_consumer"]
)

df["consumer_resolution_status"] = (
    df["resolution_status"]
    .fillna("")
    .astype(str)
    .str.upper()
)

df["consumer_resolved_fdc_id"] = (
    df["resolved_fdc_id"]
)

df["consumer_resolved_description"] = (
    df["resolved_description"]
)

df["consumer_is_resolved"] = (
    df["consumer_resolution_status"]
    .eq("RESOLVED")
)

df["consumer_is_ambiguous"] = (
    df["consumer_resolution_status"]
    .eq("AMBIGUOUS")
)

df["consumer_is_ingredient"] = (
    df["is_application_ingredient"]
    .map(bool_value)
)

df["consumer_application_ready"] = (
    df["application_ready"]
    .map(bool_value)
)


# =============================================================================
# BUILD SEARCH TOKEN CACHE
# =============================================================================

header(
    "BUILDING SEARCH TOKEN CACHE"
)

df["_search_tokens"] = (
    df["consumer_description"]
    .fillna("")
    .map(token_set)
)

print(
    "Search token cache: PASS"
)


# =============================================================================
# BUILD FDC CONSUMER LOOKUP
# =============================================================================

header(
    "BUILDING FDC CONSUMER LOOKUP"
)

consumer_lookup = df[
    [
        "fdc_id",
        "description",
        "description_normalized",
        "food_class",
        "resolved_fdc_id",
        "resolved_description",
        "resolution_status",
        "resolution_score",
        "resolution_confidence",
        "resolution_reason",
        "application_ready",
        "consumer_is_resolved",
        "consumer_is_ambiguous",
        "consumer_is_ingredient",
    ]
].copy()

consumer_lookup = (
    consumer_lookup
    .sort_values("fdc_id")
    .reset_index(drop=True)
)

print(
    f"Consumer lookup records: "
    f"{len(consumer_lookup):,}"
)

if consumer_lookup["fdc_id"].duplicated().any():

    raise SystemExit(
        "Consumer lookup contains duplicate FDC IDs."
    )

print(
    "Duplicate FDC IDs: 0"
)

print(
    "FDC lookup: PASS"
)


# =============================================================================
# BUILD DESCRIPTION LOOKUP
# =============================================================================

header(
    "BUILDING DESCRIPTION LOOKUP"
)

description_lookup = df[
    [
        "description_normalized_consumer",
        "fdc_id",
        "description",
        "food_class",
        "resolution_status",
        "resolved_fdc_id",
        "resolved_description",
        "application_ready",
        "is_application_ingredient",
    ]
].copy()

description_lookup = (
    description_lookup
    .rename(
        columns={
            "description_normalized_consumer":
                "description_normalized"
        }
    )
)

description_lookup = description_lookup[
    description_lookup[
        "description_normalized"
    ].ne("")
]

description_lookup = (
    description_lookup
    .sort_values(
        [
            "description_normalized",
            "fdc_id",
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Description lookup records: "
    f"{len(description_lookup):,}"
)

print(
    f"Unique normalized descriptions: "
    f"{description_lookup['description_normalized'].nunique():,}"
)


# =============================================================================
# BUILD INGREDIENT-ONLY TABLE
# =============================================================================

header(
    "BUILDING APPLICATION INGREDIENT TABLE"
)

ingredient_df = df[
    df["consumer_is_ingredient"]
].copy()

ingredient_df = (
    ingredient_df
    .sort_values("fdc_id")
    .reset_index(drop=True)
)

print(
    f"Ingredient application records: "
    f"{len(ingredient_df):,}"
)


# =============================================================================
# BUILD RESOLVED-ONLY TABLE
# =============================================================================

header(
    "BUILDING RESOLVED APPLICATION TABLE"
)

resolved_df = df[
    df["consumer_is_resolved"]
].copy()

resolved_df = (
    resolved_df
    .sort_values("fdc_id")
    .reset_index(drop=True)
)

print(
    f"Resolved application records: "
    f"{len(resolved_df):,}"
)


# =============================================================================
# BUILD AMBIGUOUS TABLE
# =============================================================================

header(
    "BUILDING AMBIGUOUS APPLICATION TABLE"
)

ambiguous_df = df[
    df["consumer_is_ambiguous"]
].copy()

ambiguous_df = (
    ambiguous_df
    .sort_values("fdc_id")
    .reset_index(drop=True)
)

print(
    f"Ambiguous application records: "
    f"{len(ambiguous_df):,}"
)


# =============================================================================
# BUILD QUERY INDEX
# =============================================================================

header(
    "BUILDING QUERY INDEX"
)

query_rows = []

for _, row in df.iterrows():

    resolved_id = row[
        "resolved_fdc_id"
    ]

    if pd.isna(resolved_id):

        resolved_id_value = None

    else:

        resolved_id_value = int(
            resolved_id
        )

    query_rows.append({

        "query_normalized":
            normalize_text(
                row["description"]
            ),

        "fdc_id":
            int(row["fdc_id"]),

        "description":
            row["description"],

        "food_class":
            row["food_class"],

        "resolution_status":
            row["resolution_status"],

        "resolved_fdc_id":
            resolved_id_value,

        "resolved_description":
            row["resolved_description"],

        "application_ready":
            bool_value(
                row["application_ready"]
            ),

        "is_application_ingredient":
            bool_value(
                row[
                    "is_application_ingredient"
                ]
            ),
    })


query_index_df = pd.DataFrame(
    query_rows
)

query_index_df = (
    query_index_df
    .sort_values(
        [
            "query_normalized",
            "fdc_id",
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Query index records: "
    f"{len(query_index_df):,}"
)


# =============================================================================
# CONSUMER SEARCH ENGINE
# =============================================================================

def consumer_search(
    query,
    ingredient_only=False,
    resolved_only=False,
    limit=10,
):
    """
    Application-facing food search.

    Supports:

        1. Exact normalized description
        2. Exact phrase
        3. Query tokens in any order
        4. Partial token overlap
        5. Ingredient filtering
        6. Resolved filtering
        7. Relevance ranking

    Example:

        "ground cinnamon"

    can match:

        "Spices, cinnamon, ground"

    The authoritative USDA description is never changed.
    """

    q = normalize_text(
        query
    )

    if not q:

        return df.iloc[
            0:0
        ].copy()

    query_tokens = token_set(
        q
    )

    if not query_tokens:

        return df.iloc[
            0:0
        ].copy()

    work = df

    # -------------------------------------------------------------------------
    # FILTER INGREDIENTS
    # -------------------------------------------------------------------------

    if ingredient_only:

        work = work[
            work["consumer_is_ingredient"]
        ]

    # -------------------------------------------------------------------------
    # FILTER RESOLVED
    # -------------------------------------------------------------------------

    if resolved_only:

        work = work[
            work["consumer_is_resolved"]
        ]

    if len(work) == 0:

        return work.copy()

    # -------------------------------------------------------------------------
    # SCORE RECORDS
    # -------------------------------------------------------------------------

    scores = []
    matched_flags = []
    exact_flags = []
    all_tokens_flags = []

    for _, row in work.iterrows():

        description = row[
            "consumer_description"
        ]

        description_normalized = (
            row[
                "consumer_description_normalized"
            ]
        )

        description_tokens = (
            row["_search_tokens"]
        )

        matched_tokens = (
            query_tokens
            .intersection(
                description_tokens
            )
        )

        matched = (
            len(matched_tokens) > 0
        )

        all_tokens = (
            query_tokens.issubset(
                description_tokens
            )
        )

        exact = (
            description_normalized == q
        )

        if matched:

            score = calculate_query_score(
                q,
                description,
                row[
                    "consumer_resolution_status"
                ],
            )

        else:

            score = 0.0

        # Ingredient bonus.
        if (
            matched
            and row[
                "consumer_is_ingredient"
            ]
        ):
            score += 15

        scores.append(
            score
        )

        matched_flags.append(
            matched
        )

        exact_flags.append(
            exact
        )

        all_tokens_flags.append(
            all_tokens
        )

    work = work.copy()

    work["_query_score"] = (
        scores
    )

    work["_query_matched"] = (
        matched_flags
    )

    work["_query_exact"] = (
        exact_flags
    )

    work["_all_query_tokens"] = (
        all_tokens_flags
    )

    # -------------------------------------------------------------------------
    # KEEP MATCHES
    # -------------------------------------------------------------------------

    results = work[
        work["_query_matched"]
    ].copy()

    if len(results) == 0:

        return results

    # -------------------------------------------------------------------------
    # RANKING
    # -------------------------------------------------------------------------

    results["_resolved_rank"] = (
        ~results[
            "consumer_is_resolved"
        ]
    ).astype(int)

    results["_ingredient_rank"] = (
        ~results[
            "consumer_is_ingredient"
        ]
    ).astype(int)

    results = (
        results
        .sort_values(
            [
                "_query_score",
                "_query_exact",
                "_all_query_tokens",
                "_resolved_rank",
                "_ingredient_rank",
                "fdc_id",
            ],
            ascending=[
                False,
                False,
                False,
                True,
                True,
                True,
            ],
        )
        .head(limit)
        .copy()
    )

    # -------------------------------------------------------------------------
    # REMOVE INTERNAL SEARCH COLUMNS
    # -------------------------------------------------------------------------

    results = results.drop(
        columns=[
            "_search_tokens",
            "_query_score",
            "_query_matched",
            "_query_exact",
            "_all_query_tokens",
            "_resolved_rank",
            "_ingredient_rank",
        ],
        errors="ignore",
    )

    return results


# =============================================================================
# CONSUMER QUERY TESTS
# =============================================================================

header(
    "RUNNING CONSUMER QUERY TESTS"
)

TEST_QUERIES = [

    "cinnamon",

    "ground cinnamon",

    "turmeric",

    "fresh basil",

    "dried basil",

    "olive oil",

    "oil olive",

    "baking powder",

    "jackfruit",

    "raw jackfruit",

    "kelp",

    "seaweed kelp",

    "pork shoulder",

    "black pepper",

    "garlic",

    "ginger",

    "rice",

    "flour",

    "sugar",
]


test_results = []

for query in TEST_QUERIES:

    results = consumer_search(
        query,
        ingredient_only=False,
        resolved_only=False,
        limit=5,
    )

    if len(results) == 0:

        test_pass = False

        top_fdc = None

        top_description = None

        top_status = None

    else:

        top = results.iloc[0]

        test_pass = True

        top_fdc = int(
            top["fdc_id"]
        )

        top_description = (
            top["description"]
        )

        top_status = (
            top["resolution_status"]
        )

    test_results.append({

        "query":
            query,

        "query_normalized":
            normalize_text(query),

        "result_count":
            int(len(results)),

        "top_fdc_id":
            top_fdc,

        "top_description":
            top_description,

        "top_resolution_status":
            top_status,

        "test_pass":
            test_pass,
    })

    print(
        f"{query:<25} "
        f"{'PASS' if test_pass else 'FAIL'} "
        f"results={len(results):,}"
    )


test_results_df = pd.DataFrame(
    test_results
)


# =============================================================================
# VALIDATE CORE CONSUMER EXPECTATIONS
# =============================================================================

header(
    "VALIDATING CORE CONSUMER EXPECTATIONS"
)

expected_fdc = {

    "ground cinnamon":
        171320,

    "turmeric":
        172231,

    "fresh basil":
        172232,

    "dried basil":
        171317,

    "oil olive":
        171413,

    "raw jackfruit":
        174687,

    "seaweed kelp":
        168457,

    "black pepper":
        170931,

    "garlic":
        169230,

    "ginger":
        169231,
}


expectation_results = []


for query, expected_id in expected_fdc.items():

    # -------------------------------------------------------------------------
    # CHECK AUTHORITATIVE RECORD
    # -------------------------------------------------------------------------

    record = df[
        df["fdc_id"].eq(
            expected_id
        )
    ]

    record_exists = (
        len(record) == 1
    )

    # -------------------------------------------------------------------------
    # SEARCH DISCOVERY
    # -------------------------------------------------------------------------

    results = consumer_search(
        query,
        ingredient_only=False,
        resolved_only=False,
        limit=20,
    )

    found_ids = set(
        results["fdc_id"]
        .dropna()
        .astype(int)
        .tolist()
    )

    search_found = (
        expected_id in found_ids
    )

    # -------------------------------------------------------------------------
    # DESCRIPTION
    # -------------------------------------------------------------------------

    if record_exists:

        actual_description = (
            record.iloc[0][
                "description"
            ]
        )

    else:

        actual_description = None

    # -------------------------------------------------------------------------
    # FINAL RESULT
    # -------------------------------------------------------------------------

    passed = (
        record_exists
        and search_found
    )

    expectation_results.append({

        "query":
            query,

        "expected_fdc_id":
            expected_id,

        "record_exists":
            record_exists,

        "search_found":
            search_found,

        "actual_description":
            actual_description,

        "test_pass":
            passed,
    })

    print(
        f"{query:<25} "
        f"expected={expected_id} "
        f"record_exists="
        f"{'PASS' if record_exists else 'FAIL'} "
        f"search_found="
        f"{'PASS' if search_found else 'FAIL'} "
        f"overall="
        f"{'PASS' if passed else 'FAIL'}"
    )


expectation_df = pd.DataFrame(
    expectation_results
)


# =============================================================================
# ADD SEARCH ORDER TESTS
# =============================================================================

header(
    "VALIDATING WORD-ORDER-INDEPENDENT SEARCH"
)

word_order_tests = [

    (
        "ground cinnamon",
        171320,
    ),

    (
        "cinnamon ground",
        171320,
    ),

    (
        "fresh basil",
        172232,
    ),

    (
        "basil fresh",
        172232,
    ),

    (
        "dried basil",
        171317,
    ),

    (
        "basil dried",
        171317,
    ),

    (
        "raw jackfruit",
        174687,
    ),

    (
        "jackfruit raw",
        174687,
    ),

    (
        "black pepper",
        170931,
    ),

    (
        "pepper black",
        170931,
    ),

    (
        "seaweed kelp",
        168457,
    ),

    (
        "kelp seaweed",
        168457,
    ),
]


word_order_results = []


for query, expected_id in word_order_tests:

    results = consumer_search(
        query,
        ingredient_only=False,
        resolved_only=False,
        limit=20,
    )

    found_ids = set(
        results["fdc_id"]
        .dropna()
        .astype(int)
        .tolist()
    )

    passed = (
        expected_id in found_ids
    )

    word_order_results.append({

        "query":
            query,

        "expected_fdc_id":
            expected_id,

        "search_found":
            passed,

        "test_pass":
            passed,
    })

    print(
        f"{query:<25} "
        f"expected={expected_id} "
        f"{'PASS' if passed else 'FAIL'}"
    )


word_order_df = pd.DataFrame(
    word_order_results
)


# =============================================================================
# VALIDATE RELEASE COUNTS
# =============================================================================

header(
    "VALIDATING RELEASE COUNTS"
)

checks = {}


checks["records"] = (
    len(df) == 7793
)


checks["unique_fdc_ids"] = (
    df["fdc_id"]
    .nunique(dropna=True)
    == len(df)
)


checks["resolved_count"] = (
    df["consumer_is_resolved"]
    .sum()
    == 4324
)


checks["ambiguous_count"] = (
    df["consumer_is_ambiguous"]
    .sum()
    == 3469
)


checks["ingredient_count"] = (
    df["consumer_is_ingredient"]
    .sum()
    == 3833
)


checks["self_resolution"] = (
    df["self_resolution_pass"]
    .map(bool_value)
    .sum()
    == 7793
)


checks["consumer_tests"] = (
    test_results_df[
        "test_pass"
    ].all()
)


checks["expected_records"] = (
    expectation_df[
        "test_pass"
    ].all()
)


checks["word_order_search"] = (
    word_order_df[
        "test_pass"
    ].all()
)


for name, result in checks.items():

    print(
        f"{name:<30} "
        f"{'PASS' if result else 'FAIL'}"
    )


# =============================================================================
# SAVE STEP 11 OUTPUTS
# =============================================================================

header(
    "SAVING STEP 11 OUTPUT"
)

consumer_path = (
    STEP11_DIR
    / "sr_legacy_ingredient_consumer.csv"
)

consumer_lookup_path = (
    STEP11_DIR
    / "sr_legacy_ingredient_consumer_lookup.csv"
)

description_path = (
    STEP11_DIR
    / "sr_legacy_ingredient_description_lookup.csv"
)

ingredient_path = (
    STEP11_DIR
    / "sr_legacy_ingredient_consumer_ingredients.csv"
)

resolved_path = (
    STEP11_DIR
    / "sr_legacy_ingredient_consumer_resolved.csv"
)

ambiguous_path = (
    STEP11_DIR
    / "sr_legacy_ingredient_consumer_ambiguous.csv"
)

query_index_path = (
    STEP11_DIR
    / "sr_legacy_ingredient_query_index.csv"
)

tests_path = (
    STEP11_DIR
    / "sr_legacy_step11_consumer_tests.csv"
)

expectations_path = (
    STEP11_DIR
    / "sr_legacy_step11_expected_record_tests.csv"
)

word_order_path = (
    STEP11_DIR
    / "sr_legacy_step11_word_order_tests.csv"
)


# =============================================================================
# CONSUMER TABLE
# =============================================================================

consumer_columns = [

    "consumer_fdc_id",

    "consumer_description",

    "consumer_description_normalized",

    "food_class",

    "consumer_resolution_status",

    "consumer_resolved_fdc_id",

    "consumer_resolved_description",

    "consumer_is_resolved",

    "consumer_is_ambiguous",

    "consumer_is_ingredient",

    "consumer_application_ready",

    "resolution_score",

    "resolution_confidence",

    "resolution_reason",
]


consumer_df = df[
    consumer_columns
].copy()


consumer_df.to_csv(
    consumer_path,
    index=False
)


consumer_lookup.to_csv(
    consumer_lookup_path,
    index=False
)


description_lookup.to_csv(
    description_path,
    index=False
)


ingredient_df.to_csv(
    ingredient_path,
    index=False
)


resolved_df.to_csv(
    resolved_path,
    index=False
)


ambiguous_df.to_csv(
    ambiguous_path,
    index=False
)


query_index_df.to_csv(
    query_index_path,
    index=False
)


test_results_df.to_csv(
    tests_path,
    index=False
)


expectation_df.to_csv(
    expectations_path,
    index=False
)


word_order_df.to_csv(
    word_order_path,
    index=False
)


# =============================================================================
# JSONL API TEST FIXTURES
# =============================================================================

header(
    "BUILDING APPLICATION QUERY FIXTURES"
)

fixture_path = (
    STEP11_DIR
    / "sr_legacy_step11_query_fixtures.jsonl"
)


with open(
    fixture_path,
    "w",
    encoding="utf-8"
) as f:

    for query in TEST_QUERIES:

        results = consumer_search(
            query,
            ingredient_only=True,
            resolved_only=False,
            limit=5,
        )

        record = {

            "query":
                query,

            "query_normalized":
                normalize_text(query),

            "result_count":
                int(len(results)),

            "results": [],
        }


        for _, row in results.iterrows():

            resolved_id = (
                row[
                    "resolved_fdc_id"
                ]
            )

            if pd.isna(resolved_id):

                resolved_id_value = None

            else:

                resolved_id_value = int(
                    resolved_id
                )


            record[
                "results"
            ].append({

                "fdc_id":
                    int(
                        row["fdc_id"]
                    ),

                "description":
                    row[
                        "description"
                    ],

                "food_class":
                    row[
                        "food_class"
                    ],

                "resolution_status":
                    row[
                        "resolution_status"
                    ],

                "resolved_fdc_id":
                    resolved_id_value,

                "resolved_description":
                    row[
                        "resolved_description"
                    ],
            })


        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )


print(
    f"Saved: {fixture_path}"
)


# =============================================================================
# QUALITY REPORT
# =============================================================================

header(
    "BUILDING STEP 11 QUALITY REPORT"
)

report_path = (
    STEP11_DIR
    / "sr_legacy_step11_quality_report.txt"
)


with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "=" * 80
        + "\n"
    )

    f.write(
        "SR LEGACY STEP 11 — "
        "CONSUMER APPLICATION QUALITY REPORT\n"
    )

    f.write(
        "=" * 80
        + "\n\n"
    )

    f.write(
        f"Build started: "
        f"{build_started}\n"
    )

    f.write("\n")

    f.write(
        "SOURCE\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    f.write(
        f"{RELEASE_FILE}\n"
    )

    f.write("\n")

    f.write(
        "RELEASE SUMMARY\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    f.write(
        f"Records:             "
        f"{len(df):,}\n"
    )

    f.write(
        f"Unique FDC IDs:      "
        f"{df['fdc_id'].nunique():,}\n"
    )

    f.write(
        f"Resolved:            "
        f"{int(df['consumer_is_resolved'].sum()):,}\n"
    )

    f.write(
        f"Ambiguous:           "
        f"{int(df['consumer_is_ambiguous'].sum()):,}\n"
    )

    f.write(
        f"Ingredients:         "
        f"{int(df['consumer_is_ingredient'].sum()):,}\n"
    )

    f.write(
        f"Search records:      "
        f"{len(search_df):,}\n"
    )

    f.write("\n")

    f.write(
        "VALIDATION CHECKS\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    for name, result in checks.items():

        f.write(
            f"{name:<30} "
            f"{'PASS' if result else 'FAIL'}\n"
        )

    f.write("\n")

    f.write(
        "CONSUMER QUERY TEST RESULTS\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    for _, row in test_results_df.iterrows():

        f.write(
            f"{row['query']:<25} "
            f"{'PASS' if row['test_pass'] else 'FAIL'} "
            f"results={row['result_count']}\n"
        )

        f.write(
            f"  top_fdc_id: "
            f"{row['top_fdc_id']}\n"
        )

        f.write(
            f"  top_description: "
            f"{row['top_description']}\n"
        )

    f.write("\n")

    f.write(
        "CORE EXPECTED RECORD TESTS\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    for _, row in expectation_df.iterrows():

        f.write(
            f"{row['query']:<25} "
            f"expected={row['expected_fdc_id']} "
            f"record_exists="
            f"{'PASS' if row['record_exists'] else 'FAIL'} "
            f"search_found="
            f"{'PASS' if row['search_found'] else 'FAIL'} "
            f"overall="
            f"{'PASS' if row['test_pass'] else 'FAIL'}\n"
        )

        f.write(
            f"  description: "
            f"{row['actual_description']}\n"
        )

    f.write("\n")

    f.write(
        "WORD-ORDER SEARCH TESTS\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    for _, row in word_order_df.iterrows():

        f.write(
            f"{row['query']:<25} "
            f"expected={row['expected_fdc_id']} "
            f"{'PASS' if row['test_pass'] else 'FAIL'}\n"
        )

    f.write("\n")

    f.write(
        "OUTPUT TABLES\n"
    )

    f.write(
        "-" * 80
        + "\n"
    )

    output_paths = [

        consumer_path,

        consumer_lookup_path,

        description_path,

        ingredient_path,

        resolved_path,

        ambiguous_path,

        query_index_path,

        tests_path,

        expectations_path,

        word_order_path,

        fixture_path,
    ]


    for path in output_paths:

        f.write(
            f"{path.name}\n"
        )

        f.write(
            f"  SHA256: "
            f"{sha256_file(path)}\n"
        )

    f.write("\n")

    f.write(
        "=" * 80
        + "\n"
    )

    if all(checks.values()):

        f.write(
            "STATUS: PASS\n"
        )

    else:

        f.write(
            "STATUS: REVIEW\n"
        )

    f.write(
        "=" * 80
        + "\n"
    )


print(
    f"Saved: {report_path}"
)


# =============================================================================
# STEP 11 MANIFEST
# =============================================================================

header(
    "BUILDING STEP 11 MANIFEST"
)

manifest = {

    "project":
        "SR LEGACY",

    "step":
        11,

    "layer":
        "consumer_application",

    "search_engine":
        "word_order_independent_token_search",

    "source":
        str(RELEASE_FILE),

    "created_at":
        datetime.now().isoformat(
            timespec="microseconds"
        ),

    "summary": {

        "records":
            int(len(df)),

        "unique_fdc_ids":
            int(
                df["fdc_id"]
                .nunique()
            ),

        "resolved":
            int(
                df[
                    "consumer_is_resolved"
                ].sum()
            ),

        "ambiguous":
            int(
                df[
                    "consumer_is_ambiguous"
                ].sum()
            ),

        "ingredients":
            int(
                df[
                    "consumer_is_ingredient"
                ].sum()
            ),

        "search_records":
            int(len(search_df)),
    },

    "validation": {
        k: bool(v)
        for k, v in checks.items()
    },

    "outputs": {},
}


for path in [

    consumer_path,

    consumer_lookup_path,

    description_path,

    ingredient_path,

    resolved_path,

    ambiguous_path,

    query_index_path,

    tests_path,

    expectations_path,

    word_order_path,

    fixture_path,

    report_path,
]:

    manifest[
        "outputs"
    ][path.name] = {

        "bytes":
            path.stat().st_size,

        "sha256":
            sha256_file(path),
    }


manifest_path = (
    STEP11_DIR
    / "sr_legacy_step11_manifest.json"
)


with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


print(
    f"Saved: {manifest_path}"
)


# =============================================================================
# FINAL VALIDATION
# =============================================================================

header(
    "STEP 11 FINAL VALIDATION"
)

all_pass = all(
    checks.values()
)


for name, result in checks.items():

    print(
        f"{name:<30} "
        f"{'PASS' if result else 'FAIL'}"
    )


print()


if all_pass:

    print(
        "=" * 80
    )

    print(
        "SR LEGACY STEP 11 COMPLETE"
    )

    print(
        "=" * 80
    )

    print(
        "STATUS: PASS"
    )

    print()

    print(
        "Consumer/application layer is ready."
    )

    print()

    print(
        "OUTPUT DIRECTORY:"
    )

    print(
        STEP11_DIR
    )

    print()

    print(
        "Core outputs:"
    )

    print(
        "  sr_legacy_ingredient_consumer.csv"
    )

    print(
        "  sr_legacy_ingredient_consumer_lookup.csv"
    )

    print(
        "  sr_legacy_ingredient_description_lookup.csv"
    )

    print(
        "  sr_legacy_ingredient_consumer_ingredients.csv"
    )

    print(
        "  sr_legacy_ingredient_consumer_resolved.csv"
    )

    print(
        "  sr_legacy_ingredient_consumer_ambiguous.csv"
    )

    print(
        "  sr_legacy_ingredient_query_index.csv"
    )

    print(
        "  sr_legacy_step11_consumer_tests.csv"
    )

    print(
        "  sr_legacy_step11_expected_record_tests.csv"
    )

    print(
        "  sr_legacy_step11_word_order_tests.csv"
    )

    print(
        "  sr_legacy_step11_query_fixtures.jsonl"
    )

    print(
        "  sr_legacy_step11_quality_report.txt"
    )

    print(
        "  sr_legacy_step11_manifest.json"
    )

    print(
        "=" * 80
    )

else:

    print(
        "=" * 80
    )

    print(
        "SR LEGACY STEP 11 COMPLETE"
    )

    print(
        "=" * 80
    )

    print(
        "STATUS: REVIEW"
    )

    print()

    print(
        "One or more consumer-layer validations failed."
    )

    print(
        "Do not promote Step 11 yet."
    )

    print(
        "=" * 80
    )

    raise SystemExit(1)



SR LEGACY STEP 11 — RELEASE CONSUMER / APPLICATION QUERY LAYER
Build started: 2026-08-20T19:12:09.221812

STEP 10 DIRECTORY:
C:\Users\AK\Downloads\zip\sr_legacy_step10

STEP 11 OUTPUT DIRECTORY:
C:\Users\AK\Downloads\zip\sr_legacy_step11

CHECKING STEP 10 RELEASE
PASS: sr_legacy_ingredient_release.csv
PASS: sr_legacy_ingredient_release_lookup.csv
PASS: sr_legacy_ingredient_release_search.csv

LOADING RELEASE DATA
Release rows: 7,793
Lookup rows:  7,793
Search rows:  191

VALIDATING REQUIRED COLUMNS
Required columns: 13
Required columns: PASS

NORMALIZING IDENTIFIERS
FDC ID normalization: PASS
Text normalization: PASS

BUILDING CONSUMER APPLICATION FIELDS

BUILDING SEARCH TOKEN CACHE
Search token cache: PASS

BUILDING FDC CONSUMER LOOKUP
Consumer lookup records: 7,793
Duplicate FDC IDs: 0
FDC lookup: PASS

BUILDING DESCRIPTION LOOKUP
Description lookup records: 7,793
Unique normalized descriptions: 7,792

BUILDING APPLICATION INGREDIENT TABLE
Ingredient application records: 3,833

BUIL

In [ ]:
# FN